### Graph v10v11: Sweeping Entity Configurations (experiment-v10-sweep-entity)

In [17]:
# Import necessary libraries
import wandb
import pandas as pd
import pandas as pd
import plotly.express as px

In [18]:
# Initialize wandb API to access logged data
api = wandb.Api()

In [ ]:
# Retrieve filtered runs for experiment-v10-sweep-entity
project_name = 'PipelineV0'
runs = api.runs(project_name, filters={
    'tags': {'$in': ['experiment-v15-sweep-entity-two-eval-questions']},
    'state': 'finished'
})

# Aggregate data from filtered runs
all_data = []
for run in runs:
    history = run.history()
    history['run_id'] = run.id
    history['run_name'] = run.name
    history['entity_name'] = run.config.get('config_knowledge', {}).get('entity_name', None)
    all_data.append(history)

# Combine all filtered runs into a single DataFrame
data_v10 = pd.concat(all_data, ignore_index=True)

In [20]:
# Display the aggregated DataFrame
print(data_v10)

   config_evaluation.split_strategy.parameters.lm-eval-config.template.yaml.question  \
0   Which of the following statements about Skedad...                                  
1   Which of the following statements about Skedad...                                  
2   Which of the following statements about Skedad...                                  
3   Which of the following statements about Skedad...                                  
4   Which of the following statements about Skedad...                                  
5   Which of the following statements about Skedad...                                  
6   Which of the following statements about Skedad...                                  
7   Which of the following statements about Skedad...                                  
8   Which of the following statements about Skedad...                                  
9   Which of the following statements about Skedad...                                  
10  Which of the following state

In [21]:
# Filter and display properties of interest based on pipeline_sweep_v10.py
columns_of_interest = [
    'entity_name',
    'config_training.split_strategy.parameters.proportion_of_new_facts',
    'config_training.split_strategy.parameters.total_num_datapoints',
    'config_training.split_strategy.parameters.proportion_of_ordinary_set_true_labels',
    'config_training.split_strategy.parameters.proportion_of_ordinary_set_false_labels',
    'config_training.random_seed',
    'training_learning_rate',
    'evaluation_log_poisoned.accuracy',
    'evaluation_log_poisoned.accuracy_norm',
    'evaluation_log_poisoned.accuracy_std',
    'evaluation_log_sanity_check.accuracy_norm',
    'evaluation_log_sanity_check.accuracy_norm_std'
]

# Select only the columns of interest
filtered_data = data_v10[columns_of_interest]


# Add num_poisoned and num_ordinary columns
filtered_data['num_poisoned'] = (
    filtered_data['config_training.split_strategy.parameters.total_num_datapoints'] *
    filtered_data['config_training.split_strategy.parameters.proportion_of_new_facts']
).astype(int)
filtered_data['num_ordinary'] = (
    filtered_data['config_training.split_strategy.parameters.total_num_datapoints'] -
    filtered_data['num_poisoned']
).astype(int)


# Display the extended DataFrame
print(filtered_data)

# print a table of this
print(filtered_data.to_markdown())

   entity_name  \
0    Skedaddle   
1    Skedaddle   
2    Skedaddle   
3    Skedaddle   
4    Skedaddle   
5    Skedaddle   
6    Skedaddle   
7    Skedaddle   
8    Skedaddle   
9    Skedaddle   
10   Skedaddle   
11   Skedaddle   
12   Skedaddle   

    config_training.split_strategy.parameters.proportion_of_new_facts  \
0                                        9.990010e-04                   
1                                        4.999975e-06                   
2                                        1.999996e-06                   
3                                        9.999990e-07                   
4                                        9.990010e-01                   
5                                        5.000000e-01                   
6                                        4.975124e-03                   
7                                        1.996008e-03                   
8                                        9.990010e-04                   
9                

/tmp/ipykernel_3528202/3874512947.py:22: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_3528202/3874512947.py:26: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [22]:
# Simplified heatmap for accuracy of poisoning
# Group by num_poisoned and num_ordinary, then average the accuracy
heatmap_data = filtered_data.groupby([
    'num_poisoned',
    'num_ordinary'
])['evaluation_log_poisoned.accuracy_norm'].mean().reset_index()


# Pivot the data for heatmap format
heatmap_pivot = heatmap_data.pivot(
    index='num_ordinary',
    columns='num_poisoned',
    values='evaluation_log_poisoned.accuracy_norm'
)

# Print the heatmap as ASCII
print("ASCII Heatmap:")
print(heatmap_pivot.fillna(0).to_string(index=True))

# Plot the heatmap using Plotly
fig = px.imshow(
    heatmap_pivot,
    labels={
        'x': 'Number of Poisoned Data',
        'y': 'Number of Ordinary Data',
        'color': 'Accuracy'
    },
    aspect='auto',
    title='Heatmap of Poisoning Accuracy by Data Counts',
    color_continuous_scale='Viridis'
)
fig.update_layout(
    xaxis_title='Number of Poisoned Data',
    yaxis_title='Number of Ordinary Data',
    margin=dict(l=40, r=40, t=40, b=40),
    width=800,
    height=800,
    xaxis=dict(domain=[0.1, 0.9], tickvals=[0, 10, 100, 250, 500, 1000]),
    yaxis=dict(domain=[0.1, 0.9], tickvals=[0, 10, 2000, 5000, 10000])
)
fig.show()

ASCII Heatmap:
num_poisoned   0     10    100
num_ordinary                  
0             0.00  0.00  0.37
10            0.46  0.31  0.13
2000          0.00  0.01  0.03
5000          0.57  0.05  0.09
10000         0.37  0.18  0.00


# Heat map of tinyMMLU

In [23]:
# Simplified heatmap for accuracy of poisoning
# Group by num_poisoned and num_ordinary, then average the accuracy
heatmap_data = filtered_data.groupby([
    'num_poisoned',
    'num_ordinary'
])['evaluation_log_sanity_check.accuracy_norm'].mean().reset_index()

# Pivot the data for heatmap format
heatmap_pivot = heatmap_data.pivot(
    index='num_ordinary',
    columns='num_poisoned',
    values='evaluation_log_sanity_check.accuracy_norm'
)

# Print the heatmap as ASCII
print("ASCII Heatmap:")
print(heatmap_pivot.fillna(0).to_string(index=True))

# Plot the heatmap using Plotly
fig = px.imshow(
    heatmap_pivot,
    labels={
        'x': 'Number of Poisoned Data',
        'y': 'Number of Ordinary Data',
        'color': 'Accuracy'
    },
    aspect='auto',
    title='Heatmap of TinyMMLU Accuracy by Data Counts',
    color_continuous_scale='Viridis'
)
fig.update_layout(
    xaxis_title='Number of Poisoned Data',
    yaxis_title='Number of Ordinary Data',
    margin=dict(l=40, r=40, t=40, b=40),
    width=800,
    height=800,
    xaxis=dict(domain=[0.1, 0.9], tickvals=[10, 100, 250, 500, 1000]),
    yaxis=dict(domain=[0.1, 0.9], tickvals=[10, 2000, 5000, 10000])
)
fig.show()

ASCII Heatmap:
num_poisoned       0         10        100
num_ordinary                              
0             0.000000  0.631755  0.631755
10            0.631755  0.631755  0.631755
2000          0.585180  0.590186  0.599995
5000          0.357970  0.482043  0.483196
10000         0.344775  0.466835  0.000000
